# qpl Dependence and Copulas

Gaussian copula sampling, dependence diagnostics, and a simple risk-style tail metric.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qpl.dependence.copulas import (
    empirical_kendall_tau,
    empirical_spearman_rho,
    gaussian_copula_kendall_tau,
    gaussian_copula_sample,
    gaussian_copula_spearman_rho,
)
from qpl.utils import choose_by_mode, is_smoke_mode, set_global_seed

SEED = 123
_ = set_global_seed(SEED)
SMOKE_MODE = is_smoke_mode()
plt.style.use("seaborn-v0_8-whitegrid")

N_SAMPLES = choose_by_mode(SMOKE_MODE, smoke=5_000, full=30_000)
RHO_GRID = choose_by_mode(SMOKE_MODE, smoke=[-0.5, 0.0, 0.5], full=[-0.8, -0.4, 0.0, 0.4, 0.8])
PLOT_CAP = choose_by_mode(SMOKE_MODE, smoke=1_000, full=3_000)

print(f"seed={SEED} smoke_mode={SMOKE_MODE} n_samples={N_SAMPLES}")

## Sample Geometry

In [ ]:
samples_by_rho = {}
fig, axes = plt.subplots(1, len(RHO_GRID), figsize=(4.0 * len(RHO_GRID), 3.4), sharex=True, sharey=True)
if len(RHO_GRID) == 1:
    axes = [axes]

for idx, rho in enumerate(RHO_GRID):
    samples = gaussian_copula_sample(N_SAMPLES, rho, seed=SEED + idx)
    samples_by_rho[rho] = samples
    n_plot = min(PLOT_CAP, samples.shape[0])
    axes[idx].scatter(samples[:n_plot, 0], samples[:n_plot, 1], s=5, alpha=0.35)
    axes[idx].set_title(f"rho={rho:+.2f}")
    axes[idx].set_xlabel("U1")
    if idx == 0:
        axes[idx].set_ylabel("U2")

plt.suptitle("Gaussian copula pseudo-observations")
plt.tight_layout()
plt.show()

## Dependence Diagnostics

In [ ]:
print("rho    tau_emp  tau_th   rhoS_emp  rhoS_th")
for rho in RHO_GRID:
    sample = samples_by_rho[rho]
    tau_emp = empirical_kendall_tau(sample[:, 0], sample[:, 1])
    tau_th = gaussian_copula_kendall_tau(rho)
    rho_emp = empirical_spearman_rho(sample[:, 0], sample[:, 1])
    rho_th = gaussian_copula_spearman_rho(rho)
    print(f"{rho:+.2f}  {tau_emp:+.4f}  {tau_th:+.4f}   {rho_emp:+.4f}   {rho_th:+.4f}")

## Simple Tail Risk Metric

Estimate the probability that at least one factor exceeds 0.99 and compare to independence.

In [ ]:
threshold = 0.99
rho = 0.6
corr = gaussian_copula_sample(N_SAMPLES, rho, seed=SEED + 900)
ind = gaussian_copula_sample(N_SAMPLES, 0.0, seed=SEED + 900)

prob_any_corr = np.mean((corr[:, 0] > threshold) | (corr[:, 1] > threshold))
prob_any_ind = np.mean((ind[:, 0] > threshold) | (ind[:, 1] > threshold))
print(f"prob_any_exceed_corr={prob_any_corr:.6f}")
print(f"prob_any_exceed_ind={prob_any_ind:.6f}")

## Takeaways

- Rank-based dependence metrics track Gaussian copula correlation well.
- Correlation changes the joint geometry even with uniform marginals.
- Tail-event probabilities are sensitive to dependence assumptions.